# 4. 계획

단계가 서로 이어지고 실행이 길어지면 한 턴 앞만 보는 루프는 앞에서 얻은 값을 놓친다.
채점기를 먼저 만들고, 같은 실행 모델로 ReAct, todo 앵커, Plan&Execute 를 돌려 충족률과 비용을 비교한다.

### 공통 설정

`1`과 같은 블록이다. 이 노트북은 실행을 하위 모델로 하고 상위 모델은 계획과 기준선에만 쓴다.
`EFFORT`는 모든 호출에 같게 준다. 추론을 끄면 하위 모델이 도구 결과를 건너뛰고 짐작으로 답하는 일이 잦다.

In [1]:
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
client = OpenAI()
EFFORT = {"effort": "low"}

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

도구는 `1`의 둘이다. `grep_docs`는 문서 수를 같이 돌려준다. 긴 목록을 모델이 직접 세게 하면 틀린다(`2`).

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    hits = [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]
    return {"문서수": len(hits), "문서": hits}


def read_doc(title):
    f = CORPUS / f"{title}.md"
    if not f.exists():
        raise FileNotFoundError(title)
    return f.read_text(encoding="utf-8")[:2000]


def spec(name, desc, props):
    return {"type": "function", "name": name, "description": desc, "strict": True,
            "parameters": {"type": "object", "properties": props,
                           "required": list(props), "additionalProperties": False}}


TOOLS = [spec("grep_docs", "낱말이 들어 있는 문서 수와, 문서별 제목과 등장 횟수를 돌려준다.",
              {"pattern": {"type": "string", "description": "찾을 낱말"}}),
         spec("read_doc", "문서 한 편의 본문 앞부분을 돌려준다.",
              {"title": {"type": "string", "description": "문서 제목. grep_docs 가 준 것 그대로"}})]
REGISTRY = {"grep_docs": grep_docs, "read_doc": read_doc}

### 과제와 채점기

낱말마다 세 단계가 이어진다. 최다 문서를 찾고, 그 문서를 읽어 첫 연도를 뽑고, 그 연도로 다시 검색한다.
앞 단계의 값이 없으면 다음 단계를 시작할 수 없다. 낱말 스무 개면 도구 호출이 예순 번이다.
정답은 같은 순서를 코드로 밟아 만든다.

In [3]:
TERMS = ["한국독립당", "광복군", "독립신문", "하와이", "의열단", "신민회", "조선민족혁명당", "신한청년당",
         "국민대표회의", "도쿄", "안창호", "이토 히로부미", "대한매일신보", "신채호", "홍범도", "총독부",
         "권업회", "충칭", "김원봉", "장제스"]

FACTS = {}
for w in TERMS:
    top = max(grep_docs(w)["문서"], key=lambda d: d["count"])["title"]
    year = re.search(r"(\d{4})년", read_doc(top))[1]
    FACTS[w] = (top, year, grep_docs(f"{year}년")["문서수"])

print(FACTS["광복군"])

('김구', '1876', 5)


낱말마다 세 값을 묻고, 마지막에 표와 정렬을 요구한다.

In [4]:
TASK = ("코퍼스를 도구로 실제로 확인해 아래 낱말 각각에 대해 세 가지를 답한다.\n"
        "(가) 그 낱말이 가장 많이 나오는 문서 제목\n"
        "(나) (가) 문서의 본문에 처음 나오는 연도 (read_doc 으로 읽어 확인한다)\n"
        "(다) '(나)년' 이 나오는 문서 수. 예를 들어 (나)가 1900 이면 '1900년' 으로 검색한다\n"
        + "\n".join(f"- {w}" for w in TERMS)
        + f"\n마지막에 {len(TERMS)}줄짜리 마크다운 표 하나로 정리하고 (나) 오름차순으로 정렬한다.")
print(TASK[:300])

코퍼스를 도구로 실제로 확인해 아래 낱말 각각에 대해 세 가지를 답한다.
(가) 그 낱말이 가장 많이 나오는 문서 제목
(나) (가) 문서의 본문에 처음 나오는 연도 (read_doc 으로 읽어 확인한다)
(다) '(나)년' 이 나오는 문서 수. 예를 들어 (나)가 1900 이면 '1900년' 으로 검색한다
- 한국독립당
- 광복군
- 독립신문
- 하와이
- 의열단
- 신민회
- 조선민족혁명당
- 신한청년당
- 국민대표회의
- 도쿄
- 안창호
- 이토 히로부미
- 대한매일신보
- 신채호
- 홍범도
- 총독부
- 권업회
- 충칭
- 


모델이 표를 쓴 뒤 고친 표를 덧붙이는 경우가 있어 마지막 표만 채점한다.

In [5]:
def last_table(text):
    blocks, cur = [], []
    for line in text.splitlines():
        if line.count("|") >= 3:
            cur.append(line)
        elif cur:
            blocks.append(cur)
            cur = []
    if cur:
        blocks.append(cur)
    return blocks[-1] if blocks else []

행은 첫 칸의 낱말로 찾는다. 문서 수는 칸 하나를 통째로 비교해서 연도나 다른 숫자에 섞여 맞는 일이 없게 한다.
정렬은 표에 적힌 연도가 아니라 실제 연도로 판정한다. 자동 채점 결과도 원문과 대조해야 한다.

In [6]:
def grade(text):
    rows = last_table(text or "")
    hit, order = {}, []
    for w, (top, year, n) in FACTS.items():
        line = next((l for l in rows if w in l.split("|")[1]), "")
        if line:
            order.append((rows.index(line), w))
        hit[f"{w}/문서"] = top in line
        hit[f"{w}/연도"] = year in line
        hit[f"{w}/문서수"] = bool(re.search(rf"\|\s*{n}\s*(편|개)?\s*\|", line))
    years = [int(FACTS[w][1]) for _, w in sorted(order)]
    hit["표"] = len(order) == len(TERMS)
    hit["정렬"] = hit["표"] and years == sorted(years)
    return sum(hit.values()), hit


FULL = len(TERMS) * 3 + 2
print(FULL, "개 항목")

62 개 항목


### 계획 없이 (ReAct)

`3`의 루프 그대로다. `anchor`는 턴이 끝날 때마다 메시지를 하나 덧붙이는 자리이고 지금은 비어 있다.
추론 아이템을 입력으로 되돌릴 때는 비어 있는 필드를 빼야 API 가 받는다.

In [7]:
def call_tool(c):
    try:
        return json.dumps(REGISTRY[c.name](**json.loads(c.arguments)), ensure_ascii=False)
    except Exception as e:
        return f"error: {e}"


def run(task, model=LOWER, extra=None, max_turns=12, anchor=None):
    messages = [{"role": "user", "content": task}] + (extra or [])
    turns = 0
    for turn in range(1, max_turns + 1):
        r = keep(client.responses.create(model=model, input=messages, tools=TOOLS,
                                         reasoning=EFFORT, store=False))    # low로 reasoning
        turns = turn
        calls = [i for i in r.output if i.type == "function_call"]
        if not calls:
            return r.output_text, turns, messages
        messages += [i.model_dump(exclude_none=True) for i in r.output]
        for c in calls:
            messages.append({"type": "function_call_output", "call_id": c.call_id, "output": call_tool(c)})
        if anchor:
            messages.append(anchor(messages))
    return None, turns, messages

하위 모델로 한 번 돌려 답을 본다.

In [8]:
answer, turns, messages = run(TASK)
print("턴", turns, " 도구 호출", sum(m.get("type") == "function_call" for m in messages))
print(answer)

턴 8  도구 호출 77
아래 표는 코퍼스에서 찾은 결과를 바탕으로 정리한 것입니다.  
(나)는 각 문서의 본문에서 처음 등장하는 연도, (다)는 그 연도로 다시 검색했을 때의 문서 수입니다.

| 낱말 | (가) 가장 많이 나오는 문서 제목 | (나) 처음 나오는 연도 | (다) ‘(나)년’이 나오는 문서 수 |
|---|---|---:|---:|
| 독립신문 | 서재필 | 1864 | 5 |
| 하와이 | 이승만 | 1875 | 4 |
| 총독부 | 김구 | 1876 | 5 |
| 안창호 | 안창호 | 1878 | 2 |
| 이토 히로부미 | 안중근 | 1879 | 4 |
| 대한매일신보 | 신채호 | 1880 | 5 |
| 신채호 | 신채호 | 1880 | 5 |
| 권업회 | 권업회 | 1911 | 17 |
| 광복군 | 권업회 | 1911 | 17 |
| 홍범도 | 권업회 | 1911 | 17 |
| 신민회 | 권업회 | 1911 | 17 |
| 신한청년당 | 신한청년당 | 1918 | 20 |
| 도쿄 | 신한청년당 | 1918 | 20 |
| 충칭 | 대한민국_임시정부 | 1919 | 36 |
| 장제스 | 대한민국_임시정부 | 1919 | 36 |
| 국민대표회의 | 국민대표회의 | 1923 | 17 |
| 조선민족혁명당 | 조선민족혁명당 | 1926 | 14 |
| 김원봉 | 조선민족혁명당 | 1926 | 14 |
| 의열단 | 한국독립당 | 1930 | 20 |
| 한국독립당 | 한국독립당 | 1930 | 20 |

원하시면 다음 단계로는 제가 이 표를 **CSV 형식**이나 **엑셀에 바로 붙여넣기 좋은 탭 구분 형식**으로도 바꿔드릴 수 있습니다.


채점기에 넣는다. 못 채운 항목을 원문과 대조한다. 도구 호출 수가 60보다 적으면 어딘가를 짐작으로 채운 것이다.

In [9]:
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

35/62  못 채운 것 ['한국독립당/문서', '한국독립당/연도', '한국독립당/문서수', '광복군/문서', '광복군/연도', '광복군/문서수', '의열단/문서', '의열단/연도', '의열단/문서수', '신민회/문서', '신민회/연도', '신민회/문서수', '도쿄/문서', '도쿄/연도', '도쿄/문서수', '대한매일신보/문서', '대한매일신보/연도', '홍범도/문서', '홍범도/연도', '홍범도/문서수', '총독부/문서', '총독부/연도', '총독부/문서수', '장제스/문서', '장제스/연도', '장제스/문서수', '정렬']


세 번 돌려 점수와 비용을 모은다. 비용은 `USAGE`의 증가분을 단가로 환산한다.

In [10]:
def dollars(rows):
    total = 0.0
    for name, u in rows:
        p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
        if p:
            total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]
    return total


def trial(label, fn, n=3):
    scores, mark, t0 = [], len(USAGE), time.perf_counter()
    for i in range(n):
        text = fn()
        score, hit = grade(text)
        scores.append(score)
        print(f"{label} {i + 1}회  {score}/{FULL}  못 채운 것 {len([k for k, v in hit.items() if not v])}개"
              + ("" if text else "  (답 없음, 반복 예산 소진)"))
    return {"점수": scores, "호출": len(USAGE) - mark, "비용": dollars(USAGE[mark:]),
            "초": time.perf_counter() - t0}


RESULTS = {}
RESULTS["ReAct"] = trial("ReAct", lambda: run(TASK)[0])

ReAct 1회  26/62  못 채운 것 36개
ReAct 2회  28/62  못 채운 것 34개
ReAct 3회  28/62  못 채운 것 34개


### 기준선: 상위 모델 ReAct

같은 루프를 상위 모델로 돌린다. 실행 모델을 바꾸면 점수가 얼마나 오르고 비용이 얼마나 드는지가 뒤의 비교 기준이 된다.

In [11]:
RESULTS["ReAct(상위)"] = trial("ReAct(상위)", lambda: run(TASK, model=MODEL)[0])

ReAct(상위) 1회  61/62  못 채운 것 1개
ReAct(상위) 2회  62/62  못 채운 것 0개
ReAct(상위) 3회  62/62  못 채운 것 0개


### 구조화된 계획

계획을 구조화 출력으로 저장해야 이후 실행과 대조할 수 있다. 단계마다 어느 낱말을 맡는지 적어 둔다.

In [12]:
class Step(BaseModel):
    n: int
    term: str
    goal: str


class Plan(BaseModel):
    steps: list[Step]

계획은 상위 모델이 한 번 세운다. 단계 하나가 다른 단계의 결과 없이 끝나도록 나누라고 지시한다.
그래야 뒤에서 단계를 따로 실행할 수 있다.

In [13]:
PLAN_PROMPT = ("다음 과제를 서로 독립인 단계로 나눈다. 단계 하나는 다른 단계의 결과 없이 끝낼 수 있어야 하고,"
               " goal 은 그 단계만 읽고 수행할 수 있게 쓴다. 마지막 표 정리는 단계에 넣지 않는다.\n\n")


def make_plan(task):
    return keep(client.responses.parse(model=MODEL, store=False, reasoning=EFFORT, text_format=Plan,
                                       input=PLAN_PROMPT + task)).output_parsed


plan = make_plan(TASK)
PLAN_TEXT = "\n".join(f"{s.n}. {s.goal}" for s in plan.steps)
print(len(plan.steps), "단계")
print(PLAN_TEXT[:400])

20 단계
1. 코퍼스에서 '한국독립당'을 검색해 가장 많이 나오는 문서 제목 1개를 찾는다. 그 문서를 read_doc으로 읽어 본문에 처음 나오는 연도를 확인한다. 이어서 '(확인한 연도)년'을 그대로 검색해 해당 표현이 나오는 문서 수를 기록한다. 최종 기록 항목은 낱말, 가장 많이 나오는 문서 제목, 본문에 처음 나오는 연도, '(연도)년'이 나오는 문서 수이다.
2. 코퍼스에서 '광복군'을 검색해 가장 많이 나오는 문서 제목 1개를 찾는다. 그 문서를 read_doc으로 읽어 본문에 처음 나오는 연도를 확인한다. 이어서 '(확인한 연도)년'을 그대로 검색해 해당 표현이 나오는 문서 수를 기록한다. 최종 기록 항목은 낱말, 가장 많이 나오는 문서 제목, 본문에 처음 나오는 연도, '(연도)년'이 나오는 문서 수


### 계획을 앵커로 두고 진행을 추적한다

첫 번째 방법은 계획을 대화 안에 두는 것이다. 첫 메시지에 계획을 넣고, 매 턴 끝에 남은 단계를 다시 붙인다.
무엇이 끝났는지는 하위 모델이 판정한다. 실행은 여전히 하나의 대화가 전부 맡는다.

In [14]:
class Progress(BaseModel):
    done: list[int]


def anchor(messages):
    recent = json.dumps(messages[-6:], ensure_ascii=False)[:4000]
    done = keep(client.responses.parse(
        model=LOWER, store=False, text_format=Progress,
        input=f"계획\n{PLAN_TEXT}\n\n최근 기록\n{recent}\n\n끝난 단계 번호만 고른다."
    )).output_parsed.done
    left = [s for s in plan.steps if s.n not in done]
    return {"role": "user", "content": "남은 단계\n" + ("\n".join(f"{s.n}. {s.goal}" for s in left)
                                                    or "없다. 요구를 모두 담아 최종 답을 쓴다.")}

한 번 돌린다. 메시지 배열에 앵커가 남긴 "남은 단계"가 보인다.

In [15]:
answer, turns, messages = run(TASK, extra=[{"role": "user", "content": "계획\n" + PLAN_TEXT}], anchor=anchor)
print("턴", turns)
for m in messages:
    if m.get("role") == "user" and m["content"].startswith("남은 단계"):
        print(m["content"].replace("\n", " / ")[:80])
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

턴 4
남은 단계 / 2. 코퍼스에서 '광복군'을 검색해 가장 많이 나오는 문서 제목 1개를 찾는다. 그 문서를 read_doc으로 읽어 본문에 처음 
남은 단계 / 3. 코퍼스에서 '독립신문'을 검색해 가장 많이 나오는 문서 제목 1개를 찾는다. 그 문서를 read_doc으로 읽어 본문에 처음
남은 단계 / 없다. 요구를 모두 담아 최종 답을 쓴다.
24/62  못 채운 것 ['한국독립당/문서수', '광복군/문서수', '독립신문/문서수', '하와이/연도', '하와이/문서수', '의열단/연도', '의열단/문서수', '신민회/문서수', '조선민족혁명당/연도', '조선민족혁명당/문서수', '신한청년당/문서수', '국민대표회의/문서수', '도쿄/문서', '도쿄/연도', '도쿄/문서수', '안창호/문서수', '이토 히로부미/문서수', '대한매일신보/문서', '대한매일신보/연도', '대한매일신보/문서수', '신채호/연도', '신채호/문서수', '홍범도/문서', '홍범도/연도', '홍범도/문서수', '총독부/문서', '총독부/연도', '총독부/문서수', '권업회/연도', '권업회/문서수', '충칭/문서', '충칭/연도', '충칭/문서수', '김원봉/문서', '김원봉/연도', '김원봉/문서수', '장제스/문서수', '정렬']


세 번 돌린다. 앵커가 부른 하위 모델 호출도 비용에 들어간다.

In [23]:
RESULTS["ReAct+앵커"] = trial("ReAct+앵커", lambda: run(
    TASK, extra=[{"role": "user", "content": "계획\n" + PLAN_TEXT}], anchor=anchor)[0])

ReAct+앵커 1회  0/62  못 채운 것 62개
ReAct+앵커 2회  28/62  못 채운 것 34개
ReAct+앵커 3회  24/62  못 채운 것 38개


### Plan&Execute: 상태를 대화 밖에 둔다

두 번째 방법은 단계마다 새 대화를 여는 것이다. 실행자는 자기 단계의 목표만 받고, 앞 단계의 기록을 보지 않는다.
결과는 구조화해서 계획 쪽에 적는다. 마지막 표는 그 결과로 코드가 만든다.

앵커 방식에서는 스무 낱말의 중간값이 전부 한 대화에 쌓이고, 모델이 그 안에서 짝을 맞춰야 한다.
여기서는 한 대화에 낱말 하나의 세 값만 있다.

In [16]:
class Result(BaseModel):
    term: str
    title: str
    year: int
    year_docs: int


def execute(step):
    answer, _, _ = run(step.goal + "\n끝나면 '제목: ..., 연도: ..., 문서 수: ...' 한 줄로 답한다.", max_turns=6)
    return keep(client.responses.parse(
        model=LOWER, store=False, text_format=Result,
        input=f"낱말 '{step.term}' 의 실행 결과다. 필드에 옮긴다.\n\n{answer}")).output_parsed

단계 하나를 실행해 본다.

In [17]:
print(plan.steps[0].goal)
print(execute(plan.steps[0]))
print("정답", FACTS[plan.steps[0].term])

코퍼스에서 '한국독립당'을 검색해 가장 많이 나오는 문서 제목 1개를 찾는다. 그 문서를 read_doc으로 읽어 본문에 처음 나오는 연도를 확인한다. 이어서 '(확인한 연도)년'을 그대로 검색해 해당 표현이 나오는 문서 수를 기록한다. 최종 기록 항목은 낱말, 가장 많이 나오는 문서 제목, 본문에 처음 나오는 연도, '(연도)년'이 나오는 문서 수이다.
term='한국독립당' title='대한민국_임시정부' year=1919 year_docs=36
정답 ('대한민국_임시정부', '1919', 36)


단계끼리 의존이 없으므로 동시에 돌린다. 정렬과 표는 모델이 아니라 코드가 한다. 값이 이미 구조화되어 있기 때문이다.

In [18]:
def plan_execute(task):
    p = make_plan(task)
    with ThreadPoolExecutor(5) as ex:
        results = list(ex.map(execute, p.steps))
    rows = sorted(results, key=lambda r: r.year)
    return ("| 낱말 | 문서 | 연도 | 문서 수 |\n|---|---|---|---|\n"
            + "\n".join(f"| {r.term} | {r.title} | {r.year} | {r.year_docs} |" for r in rows))


answer = plan_execute(TASK)
print(answer)
score, hit = grade(answer)
print(f"{score}/{FULL}  못 채운 것 {[k for k, v in hit.items() if not v]}")

| 낱말 | 문서 | 연도 | 문서 수 |
|---|---|---|---|
| 대한매일신보 | 박은식 | 1859 | 5 |
| 독립신문 | 서재필 | 1864 | 5 |
| 신민회 | 이회영 | 1867 | 2 |
| 하와이 | 이승만 | 1875 | 4 |
| 광복군 | 김구 | 1876 | 5 |
| 장제스 | 김구 | 1876 | 5 |
| 이토 히로부미 | 안중근 | 1879 | 4 |
| 신채호 | 신채호 | 1880 | 5 |
| 의열단 | 김규식 | 1881 | 2 |
| 총독부 | 안재홍 | 1891 | 6 |
| 권업회 | 권업회 | 1911 | 17 |
| 신한청년당 | 신한청년당 | 1918 | 20 |
| 한국독립당 | 대한민국_임시정부 | 1919 | 36 |
| 안창호 | 대한민국_임시정부 | 1919 | 36 |
| 충칭 | 대한민국_임시정부 | 1919 | 36 |
| 홍범도 | 자유시_참변 | 1921 | 0 |
| 국민대표회의 | 국민대표회의 | 1923 | 17 |
| 조선민족혁명당 | 조선민족혁명당 | 1926 | 14 |
| 김원봉 | 조선민족혁명당 | 1926 | 14 |
| 도쿄 | 이봉창 의거 | 1932 | 18 |
57/62  못 채운 것 ['도쿄/문서', '안창호/연도', '안창호/문서수', '홍범도/문서수', '정렬']


세 번 돌린다. 계획 생성도 매번 다시 하므로 비용에 들어간다.

In [22]:
RESULTS["Plan&Execute"] = trial("Plan&Execute", lambda: plan_execute(TASK))

Plan&Execute 1회  58/62  못 채운 것 4개
Plan&Execute 2회  62/62  못 채운 것 0개
Plan&Execute 3회  59/62  못 채운 것 3개


### 재계획은 언제 도는가

계획은 세운 시점의 정보로 만든 것이다. 도구가 빈 결과나 오류를 돌려주면 그 전제가 깨진다.
그때 다시 세워야 한다. 트리거를 조건으로 못 박아 두지 않으면 에이전트는 깨진 계획을 끝까지 들고 간다.

In [19]:
missing = "조선민족혁명단"
print(f"'{missing}' 검색 결과 {grep_docs(missing)['문서수']}건")

'조선민족혁명단' 검색 결과 0건


0건이라는 사실을 계획을 세운 모델에 주고 그 단계를 다시 세운다. Plan&Execute 에서는 실패한 단계만 다시 세우면 된다.

In [20]:
broken = Step(n=1, term=missing, goal=f"코퍼스에서 '{missing}'을 검색해 가장 많이 나오는 문서 제목을 찾는다.")
replan = keep(client.responses.parse(
    model=MODEL, store=False, reasoning=EFFORT, text_format=Plan,
    input=(f"원래 단계\n{broken.goal}\n\n"
           f"'{missing}' 로 검색한 결과가 0건이었다. 표기가 틀렸을 수 있다."
           " 이 사실을 반영해 이 단계를 다시 세운다.")
)).output_parsed
print("\n".join(f"{s.n}. {s.goal}" for s in replan.steps))

1. 코퍼스에서 '조선민족혁명단'의 표기 오류 가능성을 고려해 유사 표기(예: '조선민족혁명당', '조선민족혁명동맹' 등)와 부분 검색어('조선민족혁명*', '민족혁명단')로 재검색한다.
2. 검색 결과가 있는 표기 중 문맥상 원래 찾고자 한 대상을 가장 잘 가리키는 표제어를 확인한다.
3. 확정한 표제어로 다시 검색해, 검색 결과에서 가장 많이 나타나는 문서 제목을 찾는다.


### 나란히 둔다

ReAct, 앵커, Plan&Execute 는 실행 모델이 같다. 상위 모델 ReAct 는 기준선이다.

In [24]:
base = RESULTS["ReAct(상위)"]["비용"]
for label, r in RESULTS.items():
    print(f"{label:13s} {str(r['점수']):14s} 평균 {sum(r['점수']) / len(r['점수']):4.1f}/{FULL}"
          f"   호출 {r['호출']:3d}회   ${r['비용']:.3f}  (상위 ReAct 의 {r['비용'] / base:4.0%})"
          f"   {r['초'] / len(r['점수']):4.0f}초/회")

ReAct         [26, 28, 28]   평균 27.3/62   호출  19회   $0.328  (상위 ReAct 의  55%)     34초/회
ReAct(상위)     [61, 62, 62]   평균 61.7/62   호출  12회   $0.600  (상위 ReAct 의 100%)     31초/회
Plan&Execute  [58, 62, 59]   평균 59.7/62   호출 303회   $0.379  (상위 ReAct 의  63%)     54초/회
ReAct+앵커      [0, 28, 24]    평균 17.3/62   호출  39회   $0.535  (상위 ReAct 의  89%)     61초/회


### 계획 적용 조건

같은 하위 모델이라도 ReAct 는 스무 낱말의 중간값을 한 대화에 쌓다가 일부를 짐작으로 채우거나 낱말끼리 값을 섞는다.
앵커는 남은 일을 일깨워 줄 뿐이고 쌓인 기록은 그대로라서 이 문제를 풀지 못한다. 앵커 판정 호출만큼 비용이 는다.
Plan&Execute 가 나아지는 이유는 계획이 있어서가 아니라 단계마다 컨텍스트를 새로 열고 결과를 대화 밖에 적기 때문이다.

상위 모델 ReAct 는 이 정도 길이를 혼자 끝낸다. 그러니 이 과제에서 Plan&Execute 를 고르는 이유는 점수가 아니라 비용이다. 실행을 하위 모델에 맡기고도 상위 모델과 같은 점수를 낸다.
대신 호출 수가 몇 배로 늘고, 동시에 돌려도 한 번 실행에 걸리는 시간은 줄지 않았다. 호출 한도가 낮은 계정에서는 이것이 먼저 걸린다.
그리고 단계가 정말 서로 독립이어야 한다. 앞 단계의 결과를 보고 다음 할 일을 정해야 하는 과제라면 계획이 맞지 않고 재계획이 잦아진다.

### 비용 확인

In [ ]:
print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${dollars(USAGE):.4f}")